# Adım 5 — Score Aggregation

Adım 4'ün ürettiği 5 anomali skorunu **tek bir `fraud_score`** altında birleştirir.  
Girdi: `step4_scored.parquet` (590,540 × 509)  
Çıktı: `df_final_train.parquet` (590,540 × 510) + `weight_report_train.json`

---

## Temel Soru: Hangi Katmana Ne Kadar Güvenilmeli?

Adım 4'te 5 farklı anomali sinyali üretildi. Bunları toplamak için iki strateji var:

| Strateji | Açıklama | Ne zaman? |
|----------|----------|-----------|
| **Default ağırlıklar** | Önceden tanımlı sabit ağırlıklar | `target_col=None` (test seti) |
| **AUC bazlı otomatik** | Her katmanın `isFraud` ayırt etme gücüne göre | `target_col="isFraud"` (train seti) |
| **Manuel override** | `weights` parametresiyle dışarıdan verilir | Özel senaryolar |

**Train seti AUC sonuçları:**

| Katman | AUC | Ağırlık | Katkı Payı |
|--------|-----|---------|-----------|
| `column_score_mean` | **0.7309** | **%37.1** | **%37.4** |
| `column_score_max` | 0.7024 | %32.5 | %32.4 |
| `multivariate_score` | 0.6413 | %22.7 | %21.5 |
| `entity_score` | 0.5302 | %4.85 | %4.9 |
| `temporal_score` | 0.5178 | %2.86 | %3.8 |

> **Kritik bulgu:** `column_score_mean` en güçlü sinyal (%37 ağırlık).  
> Column bazlı sinyaller toplam ağırlığın **%69.8**'ini oluşturuyor.

## Fonksiyonlar ve Tasarım

### `robust_normalize(series)`

**Neden robust?** Standart min-max normalize outlier'lara karşı kırılgandır —  
tek bir aşırı değer tüm skalayı sıkıştırır. IQR bazlı yaklaşım daha dayanıklıdır.

```
normalize(x) = (x - median) / IQR   →   [0, 1] clip
```

- NaN değerler önce 0 ile doldurulur  
- IQR = 0 ise (sabit kolon) tüm değerler 0 döner

### `compute_auc_weights(df, score_cols, target_col)`

Her katmanın `isFraud` üzerindeki ayırt edici gücünü ölçer.

**Dönüşüm:**
```
AUC → (AUC - 0.5) → normalize et → ağırlık
```

- AUC = 0.5 (rastgele tahmin) → ağırlık = 0
- AUC < 0.5 → 0'a çekiliyor (ters çevirme yok — güvensiz sinyal)
- AUC = 0.73 gibi güçlü sinyaller → yüksek ağırlık

### `normalize_scores(df, score_cols)`

`score_cols` listesindeki her kolona `robust_normalize` uygular.  
Orijinal DataFrame'i değiştirmez, normalize edilmiş kopyayı döner.

### `aggregate_scores(df, score_cols, weights, target_col, output_col)`

Ana orkestratör. 4 adımda çalışır:

```
1. robust_normalize()        → her skor [0,1]'e çekilir
2. Ağırlık belirleme         → auc / default / manual
3. Weighted sum              → fraud_score = Σ (skor × ağırlık)
4. Katkı payı hesabı         → raporlama için her katmanın payı
```

**Dönen değer:** `(df_out, weight_report)`  
`weight_report` içeriği: `mode`, `weights`, `auc_scores`, `contribution`

In [1]:
# ─────────────────────────────────────────────
# Adım 5 — Score Aggregation
# Input : df_scored (Adım 4 çıktısı)
# Output: df_scored + "fraud_score" kolonu
#         + weight_report sözlüğü
# ─────────────────────────────────────────────

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from typing import Optional


# ── Varsayılan ağırlıklar ────────────────────────────────────────────────────
# Gerekçe:
#  • multivariate  : Isolation Forest tüm feature uzayını görür → en yüksek
#  • entity        : fraud literatüründe kart/hesap bazlı örüntü en güçlü sinyal
#  • column        : tek boyutlu z-score; max sürümü ani spike'ları yakalar → orta
#  • temporal      : önemli ama veri kalitesine bağımlı → daha düşük
#  • column_mean   : gürültüye karşı yumuşak ortalama → destek rolü
DEFAULT_WEIGHTS = {
    "multivariate_score" : 0.30,
    "entity_score"       : 0.25,
    "column_score_max"   : 0.20,
    "temporal_score"     : 0.15,
    "column_score_mean"  : 0.10,
}


def robust_normalize(series: pd.Series) -> pd.Series:
    """
    IQR bazlı normalize: (x - median) / IQR, ardından [0,1] clip.
    NaN değerler önce 0 ile doldurulur, sonra normalize edilir.
    """
    series = series.fillna(0.0)          

    q25 = series.quantile(0.25)
    q75 = series.quantile(0.75)
    iqr  = q75 - q25

    if iqr == 0:
        return pd.Series(0.0, index=series.index)

    normalized = (series - series.median()) / iqr
    return normalized.clip(0, 1)


def compute_auc_weights(
    df: pd.DataFrame,
    score_cols: list[str],
    target_col: str,
) -> dict[str, float]:
    """
    Her skor kolonu için AUC hesapla → normalize ederek ağırlığa çevir.
    
    AUC = 0.5 (rastgele) → o katmana ağırlık verme.
    AUC < 0.5 → ters sinyalli skor; yine de 0 ağırlık ver (ters çevirme yok).
    """
    aucs = {}
    y_true = df[target_col]

    for col in score_cols:
        try:
            auc = roc_auc_score(y_true, df[col])
            # 0.5'in altını 0'a çek: negatif sinyal → ağırlıklandırma dışı
            aucs[col] = max(0.0, auc - 0.5)
        except Exception:
            aucs[col] = 0.0

    total = sum(aucs.values())
    if total == 0:
        # Hiçbir skor ayırt edici değil → default'a dön (çağıran karar verir)
        return {}

    return {col: val / total for col, val in aucs.items()}


def normalize_scores(
    df: pd.DataFrame,
    score_cols: list[str],
) -> pd.DataFrame:
    """
    score_cols listesindeki her kolonu robust_normalize ile [0,1]'e çeker.
    Orijinal df'yi değiştirmez; normalize edilmiş kopyayı döner.
    """
    df_norm = df.copy()
    for col in score_cols:
        if col in df_norm.columns:
            df_norm[col] = robust_normalize(df_norm[col])
        else:
            # Eksik kolon → sıfır doldur, uyar
            print(f"[WARN] normalize_scores: '{col}' bulunamadı, 0 ile dolduruldu.")
            df_norm[col] = 0.0
    return df_norm


def aggregate_scores(
    df: pd.DataFrame,
    score_cols: Optional[list[str]] = None,
    weights: Optional[dict[str, float]] = None,
    target_col: Optional[str] = None,
    output_col: str = "fraud_score",
) -> tuple[pd.DataFrame, dict]:
    """
    Adım 4 çıktısındaki anomali skorlarını normalize edip tek fraud_score üretir.

    Parametre
    ---------
    df          : Adım 4'ten gelen df_scored
    score_cols  : Birleştirilecek skor kolonları; None → DEFAULT_WEIGHTS anahtarları
    weights     : Manuel ağırlık sözlüğü; None → otomatik seçim
    target_col  : Eğitim setinde "isFraud"; test setinde None geç
    output_col  : Çıktı kolonu adı (varsayılan "fraud_score")

    Döner
    -----
    df_out        : df + output_col kolonu
    weight_report : {
        "mode"         : "auc" | "default" | "manual",
        "weights"      : {col: float},
        "auc_scores"   : {col: float} | None,
        "contribution" : {col: float},  # her katmanın final skordaki katkı payı
    }
    """
    if score_cols is None:
        score_cols = list(DEFAULT_WEIGHTS.keys())

    # 1. Robust normalize
    df_norm = normalize_scores(df, score_cols)

    # 2. Ağırlık belirleme
    mode = "default"
    auc_scores_report = None

    if weights is not None:
        # Manuel ağırlık → normalize et (toplamı 1'e tamamla)
        total = sum(weights.values())
        final_weights = {col: w / total for col, w in weights.items()
                         if col in score_cols}
        mode = "manual"

    elif target_col is not None and target_col in df.columns:
        # AUC bazlı otomatik ağırlık
        auc_weights = compute_auc_weights(df_norm, score_cols, target_col)

        if auc_weights:
            final_weights = auc_weights
            mode = "auc"
            # Raporlama için ham AUC değerleri
            auc_scores_report = {}
            y_true = df[target_col]
            for col in score_cols:
                try:
                    auc_scores_report[col] = round(
                        roc_auc_score(y_true, df_norm[col]), 4
                    )
                except Exception:
                    auc_scores_report[col] = None
        else:
            # AUC hesaplanamadı → default'a düş
            final_weights = {col: DEFAULT_WEIGHTS.get(col, 0.0)
                             for col in score_cols}
            mode = "default"
            print("[INFO] AUC ağırlıkları hesaplanamadı, default ağırlıklara düşüldü.")
    else:
        # target_col yok (test seti) → default
        final_weights = {col: DEFAULT_WEIGHTS.get(col, 0.0)
                         for col in score_cols}
        mode = "default"

    # Eksik kolonlar için ağırlığı sıfırla
    for col in score_cols:
        if col not in final_weights:
            final_weights[col] = 0.0

    # Toplamı yeniden normalize et (bazı kolonlar 0 ağırlık almış olabilir)
    total_w = sum(final_weights.values())
    if total_w == 0:
        raise ValueError("Tüm ağırlıklar sıfır — skor listesini ve veriyi kontrol edin.")
    final_weights = {col: w / total_w for col, w in final_weights.items()}

    # 3. Weighted sum → fraud_score
    df_out = df.copy()
    df_out[output_col] = sum(
        df_norm[col] * final_weights.get(col, 0.0)
        for col in score_cols
        if col in df_norm.columns
    ).clip(0, 1)

    # 4. Katkı payı: her katmanın ortalama ağırlıklı katkısı
    contribution = {
        col: round(
            (df_norm[col] * final_weights.get(col, 0.0)).mean() /
            df_out[output_col].mean()
            if df_out[output_col].mean() > 0 else 0.0,
            4
        )
        for col in score_cols
        if col in df_norm.columns
    }

    weight_report = {
        "mode"        : mode,
        "weights"     : {col: round(w, 4) for col, w in final_weights.items()},
        "auc_scores"  : auc_scores_report,
        "contribution": contribution,
    }

    return df_out, weight_report

## Çalıştır

**Girdi:** `step4_scored.parquet`  
**Mod:** AUC bazlı otomatik ağırlıklandırma (`target_col="isFraud"`)

Beklenen çıktı:
```
Ağırlık modu : auc

AUC skorları:
  column_score_mean   0.7309   ← en güçlü
  column_score_max    0.7024
  multivariate_score  0.6413
  entity_score        0.5302
  temporal_score      0.5178   ← en zayıf

Ayrım oranı: 2.01x
  Fraud ortalaması     : 0.5947
  Non-fraud ortalaması : 0.2960
```

In [2]:
import json
from pathlib import Path

# ── Yollar ──────────────────────────────────────────────────────
OUTPUT_DIR = Path("outputs/step5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCORED_PATH  = Path("../Adım 4/step4_scored.parquet")   # Adım 4 çıktısı
FINAL_PATH   = OUTPUT_DIR / "df_final_train.parquet"
REPORT_PATH  = OUTPUT_DIR / "weight_report_train.json"

# ── Adım 4 çıktısını yükle ──────────────────────────────────────
df_scored = pd.read_parquet(SCORED_PATH)

print(f"Yüklendi: {df_scored.shape[0]:,} satır, {df_scored.shape[1]} kolon")

# ── Adım 5 — aggregate_scores ───────────────────────────────────
df_final, weight_report = aggregate_scores(
    df=df_scored,
    target_col="isFraud",      # train setinde mevcut
)

# ── Parquet kaydet ──────────────────────────────────────────────
df_final.to_parquet(FINAL_PATH, index=False)
print(f"Kaydedildi : {FINAL_PATH}")
print(f"Shape      : {df_final.shape}")

# ── Weight report kaydet (JSON) ─────────────────────────────────
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(weight_report, f, indent=2, ensure_ascii=False)
print(f"Kaydedildi : {REPORT_PATH}")

# ── Özet ekrana ─────────────────────────────────────────────────
print("\n── Ağırlık modu:", weight_report["mode"])
print("\n── Ağırlıklar:")
for col, w in weight_report["weights"].items():
    print(f"   {col:<25} {w:.4f}")

if weight_report["auc_scores"]:
    print("\n── AUC skorları:")
    for col, auc in weight_report["auc_scores"].items():
        print(f"   {col:<25} {auc}")

print("\n── Katman katkı payları:")
for col, c in weight_report["contribution"].items():
    print(f"   {col:<25} {c:.4f}")

print("\n── fraud_score özeti:")
print(df_final["fraud_score"].describe().round(4))

# Fraud vs non-fraud ayrımı kontrolü
if "isFraud" in df_final.columns:
    fraud_mean    = df_final.loc[df_final["isFraud"]==1, "fraud_score"].mean()
    nonfraud_mean = df_final.loc[df_final["isFraud"]==0, "fraud_score"].mean()
    print(f"\n   Fraud ortalaması     : {fraud_mean:.4f}")
    print(f"   Non-fraud ortalaması : {nonfraud_mean:.4f}")
    print(f"   Ayrım oranı          : {fraud_mean/nonfraud_mean:.2f}x")

Yüklendi: 590,540 satır, 509 kolon
Kaydedildi : outputs\step5\df_final_train.parquet
Shape      : (590540, 510)
Kaydedildi : outputs\step5\weight_report_train.json

── Ağırlık modu: auc

── Ağırlıklar:
   multivariate_score        0.2270
   entity_score              0.0485
   column_score_max          0.3251
   temporal_score            0.0286
   column_score_mean         0.3709

── AUC skorları:
   multivariate_score        0.6413
   entity_score              0.5302
   column_score_max          0.7024
   temporal_score            0.5178
   column_score_mean         0.7309

── Katman katkı payları:
   multivariate_score        0.2153
   entity_score              0.0490
   column_score_max          0.3244
   temporal_score            0.0376
   column_score_mean         0.3737

── fraud_score özeti:
count    590540.0000
mean          0.3065
std           0.3227
min           0.0000
25%           0.0298
50%           0.1601
75%           0.5680
max           1.0000
Name: fraud_score, dtyp

---

## Sonuç Yorumu

### AUC Sıralaması Adım 4 Lift Sıralamasını Doğruluyor

| Katman | Adım 4 Lift | Adım 5 AUC | Ağırlık |
|--------|------------|------------|---------|
| `column_score_mean` | 1.64x | **0.73** | %37.1 |
| `column_score_max` | 1.47x | 0.70 | %32.5 |
| `multivariate_score` | 1.26x | 0.64 | %22.7 |
| `entity_score` | 1.07x | 0.53 | %4.85 |
| `temporal_score` | 1.03x | 0.52 | %2.86 |

Lift sıralaması ile AUC sıralaması birebir örtüşüyor — Adım 4 analizi tutarlı.

### Separation: 2.01x

```
Fraud ortalaması     : 0.5947
Non-fraud ortalaması : 0.2960
Ayrım oranı          : 2.01x
```

Fraud işlemler non-fraud işlemlere göre ortalamada **2 kat** daha yüksek skor alıyor.  
Bu oran Adım 6 (Context Adjustment) ve Adım 7 (Rule Engine) ile **2.69x**'e çıkarılacak.

### Test Seti Kullanımı

```python
# Test setinde target_col=None → default ağırlıklar devreye girer
df_test_final, _ = aggregate_scores(df=df_test_scored, target_col=None)
```

**Sonraki adım:** Adım 6 — Context Adjustment Engine  
Girdi: `df_final_train.parquet`